# Day 22 Capstone: Energy-Demand Forecasting with LSTM + Attention

This notebook completes the assignment using simple code and comments.

## What this notebook does
1. Generates two years of hourly energy-demand data.
2. Converts the series into 168-hour input windows and 24-hour targets.
3. Splits data chronologically into train, validation, and test sets.
4. Creates a seasonal-naive baseline.
5. Trains a simple LSTM.
6. Trains an LSTM with attention.
7. Compares all models using MAE, RMSE, and MAPE.
8. Plots one forecast and prints a 150–250 word analysis.

Run every cell from top to bottom.

## Prerequisites: imports and synthetic energy-demand data
The data is generated inside this notebook, so no dataset or internet download is required.

In [ ]:
# Import the libraries used in this notebook.
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Fix random seeds so results are more repeatable.
RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

# Create about two years of hourly readings.
N = 24 * 7 * 104
hours = np.arange(N)

# Build realistic demand patterns.
trend = 0.0015 * hours                              # Slow upward trend
daily = 8.0 * np.sin(2 * np.pi * hours / 24 - 1.0) # Daily 24-hour cycle
weekly = 4.0 * np.sin(2 * np.pi * hours / (24 * 7))# Weekly cycle
weekend = -3.0 * ((hours // 24) % 7 >= 5)           # Lower weekend demand
noise = np.random.normal(0, 1.2, N)                 # Random variation

# Final energy-demand series.
demand = 50.0 + trend + daily + weekly + weekend + noise
demand = demand.astype("float32")

print("Series length:", len(demand))
print("Mean:", round(float(demand.mean()), 2))
print("Minimum:", round(float(demand.min()), 2))
print("Maximum:", round(float(demand.max()), 2))

In [ ]:
# Plot the first four weeks so daily and weekly patterns are visible.
plt.figure(figsize=(12, 4))
plt.plot(demand[:24 * 28])
plt.title("Energy demand - first 4 weeks")
plt.xlabel("Hour")
plt.ylabel("Demand")
plt.grid(alpha=0.3)
plt.show()

## Step 1: Create sliding windows
Each input contains the previous **168 hours (one week)**. The target contains the next **24 hours (one day)**.

In [ ]:
LOOKBACK = 168  # Number of past hours given to the model
HORIZON = 24    # Number of future hours predicted

# Convert one long time series into input-output samples.
def make_windows(series, lookback, horizon):
    X = []
    y = []

    # Move through the series one hour at a time.
    last_start = len(series) - lookback - horizon + 1
    for start in range(last_start):
        input_end = start + lookback
        output_end = input_end + horizon

        X.append(series[start:input_end])
        y.append(series[input_end:output_end])

    return np.array(X, dtype="float32"), np.array(y, dtype="float32")

X_all, y_all = make_windows(demand, LOOKBACK, HORIZON)

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)
print("One X row = 168 past hours")
print("One y row = 24 future hours")

## Step 2: Chronological split and train-only normalization
Time-series data must not be shuffled before splitting. The first 70% is training data, the next 15% is validation data, and the final 15% is test data.

The mean and standard deviation are calculated only from training inputs to avoid future-data leakage.

In [ ]:
# Find chronological split positions.
number_of_samples = len(X_all)
train_end = int(number_of_samples * 0.70)
validation_end = int(number_of_samples * 0.85)

# Split without shuffling.
X_train_raw = X_all[:train_end]
y_train_raw = y_all[:train_end]

X_val_raw = X_all[train_end:validation_end]
y_val_raw = y_all[train_end:validation_end]

X_test_raw = X_all[validation_end:]
y_test_raw = y_all[validation_end:]

# Learn normalization statistics from training inputs only.
train_mean = X_train_raw.mean()
train_std = X_train_raw.std()

# Simple functions for scaling and returning to original units.
def normalize(values):
    return (values - train_mean) / train_std

def denormalize(values):
    return values * train_std + train_mean

# Normalize every split using the same training statistics.
X_train = normalize(X_train_raw)
y_train = normalize(y_train_raw)
X_val = normalize(X_val_raw)
y_val = normalize(y_val_raw)
X_test = normalize(X_test_raw)
y_test = normalize(y_test_raw)

# LSTM expects: samples, time steps, features.
# We have one feature, energy demand, so add a final axis of size 1.
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Training mean:", round(float(train_mean), 3))
print("Training standard deviation:", round(float(train_std), 3))

## Step 3: Seasonal-naive baseline
The baseline assumes that the next 24 hours will look like the final 24 hours in the previous week. A learned model should beat this strong and simple rule.

In [ ]:
# Evaluation functions. Lower values are better.
def mae(actual, predicted):
    return float(np.mean(np.abs(actual - predicted)))

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

def mape(actual, predicted):
    # Small epsilon prevents division by zero.
    epsilon = 1e-8
    return float(np.mean(np.abs((actual - predicted) / (actual + epsilon))) * 100)

# The last 24 hours of every test input become the baseline prediction.
baseline_predictions = X_test_raw[:, -HORIZON:]
actual_test = y_test_raw

baseline_mae = mae(actual_test, baseline_predictions)
baseline_rmse = rmse(actual_test, baseline_predictions)
baseline_mape = mape(actual_test, baseline_predictions)

print("SEASONAL-NAIVE BASELINE")
print("MAE :", round(baseline_mae, 3))
print("RMSE:", round(baseline_rmse, 3))
print("MAPE:", round(baseline_mape, 2), "%")

## Step 4: Build and train a simple LSTM
The LSTM reads the 168-hour history. A Dense layer predicts all 24 future hours at once.

In [ ]:
# Build a direct multi-output LSTM model.
def build_lstm():
    model = keras.Sequential([
        keras.Input(shape=(LOOKBACK, 1)),

        # LSTM remembers useful patterns from the past week.
        layers.LSTM(48),

        # Dropout helps reduce overfitting.
        layers.Dropout(0.1),

        # Predict all 24 future hours together.
        layers.Dense(HORIZON)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )
    return model

lstm_model = build_lstm()
lstm_model.summary()

# Stop training when validation loss does not improve for 3 epochs.
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

lstm_history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2
)

print("Epochs completed:", len(lstm_history.history["loss"]))

In [ ]:
# Plot LSTM training and validation loss.
plt.figure(figsize=(8, 4))
plt.plot(lstm_history.history["loss"], label="Training loss")
plt.plot(lstm_history.history["val_loss"], label="Validation loss")
plt.title("Simple LSTM learning curve")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Step 5: Add attention over LSTM encoder states
Here, the LSTM returns a hidden state for every input hour. Self-attention allows the model to give more importance to useful hours in the previous week.

In [ ]:
# Build an LSTM model with self-attention.
def build_lstm_attention():
    inputs = keras.Input(shape=(LOOKBACK, 1))

    # Return all 168 hidden states instead of only the final state.
    encoder_states = layers.LSTM(48, return_sequences=True)(inputs)

    # Self-attention compares the encoder states with one another.
    attention_output = layers.Attention()([
        encoder_states,
        encoder_states
    ])

    # Add original states back to preserve information.
    combined = layers.Add()([encoder_states, attention_output])

    # Normalize values to make training more stable.
    combined = layers.LayerNormalization()(combined)

    # Convert all attended time steps into one vector.
    pooled = layers.GlobalAveragePooling1D()(combined)
    pooled = layers.Dropout(0.1)(pooled)

    # Predict the following 24 hours.
    outputs = layers.Dense(HORIZON)(pooled)

    model = keras.Model(inputs, outputs, name="lstm_attention")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )
    return model

attention_model = build_lstm_attention()
attention_model.summary()

attention_early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

attention_history = attention_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=[attention_early_stop],
    verbose=2
)

print("Epochs completed:", len(attention_history.history["loss"]))
print("LSTM parameters:", lstm_model.count_params())
print("LSTM + attention parameters:", attention_model.count_params())

## Step 6: Evaluate all approaches in original demand units
Predictions are converted back from normalized values. Then baseline, LSTM, and LSTM with attention are compared using the same test set.

In [ ]:
# Predict with a neural-network model and calculate all metrics.
def evaluate_model(model, model_name):
    normalized_predictions = model.predict(X_test, verbose=0)
    predictions = denormalize(normalized_predictions)

    return {
        "model": model_name,
        "MAE": mae(actual_test, predictions),
        "RMSE": rmse(actual_test, predictions),
        "MAPE": mape(actual_test, predictions),
        "predictions": predictions
    }

# Store the baseline results.
baseline_result = {
    "model": "Seasonal naive",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "MAPE": baseline_mape,
    "predictions": baseline_predictions
}

# Evaluate both trained models.
lstm_result = evaluate_model(lstm_model, "LSTM")
attention_result = evaluate_model(attention_model, "LSTM + attention")

results = [baseline_result, lstm_result, attention_result]

# Print a clear metric table.
print(f"{'Model':20s} {'MAE':>10s} {'RMSE':>10s} {'MAPE':>10s}")
print("-" * 54)

for result in results:
    print(
        f"{result['model']:20s} "
        f"{result['MAE']:10.3f} "
        f"{result['RMSE']:10.3f} "
        f"{result['MAPE']:9.2f}%"
    )

# Best model is the one with the lowest MAE.
best_result = min(results, key=lambda item: item["MAE"])
print("
Lowest MAE:", best_result["model"])

## Step 7: Plot one 24-hour test forecast
The plot compares the real demand, seasonal-naive prediction, and the better neural network.

In [ ]:
# Choose the better of the two trained neural networks using MAE.
if attention_result["MAE"] <= lstm_result["MAE"]:
    best_network = attention_result
else:
    best_network = lstm_result

# Plot the first test window.
sample_number = 0
truth = actual_test[sample_number]
naive_forecast = baseline_predictions[sample_number]
network_forecast = best_network["predictions"][sample_number]
forecast_hours = np.arange(1, HORIZON + 1)

plt.figure(figsize=(10, 5))
plt.plot(forecast_hours, truth, marker="o", label="Actual demand")
plt.plot(forecast_hours, naive_forecast, marker="s", label="Seasonal naive")
plt.plot(forecast_hours, network_forecast, marker="^", label=best_network["model"])
plt.title("24-hour energy-demand forecast")
plt.xlabel("Hours ahead")
plt.ylabel("Demand")
plt.xticks(np.arange(1, 25, 2))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("First 6 actual values:", np.round(truth[:6], 2))
print("First 6 baseline values:", np.round(naive_forecast[:6], 2))
print("First 6 network values:", np.round(network_forecast[:6], 2))

### Teacher forcing versus autoregressive forecasting

This notebook uses a **direct multi-output forecast**. The model reads the 168-hour input once and predicts all 24 future hours together. Therefore, teacher forcing is not used during training or inference.

In an autoregressive sequence-to-sequence model, the next forecast depends on earlier forecast values. During training, teacher forcing may supply the real previous value, but inference must use the model's own previous prediction. This difference can cause **exposure bias**, and mistakes may accumulate step by step.

## Step 8: Final written analysis
The following cell creates a 150–250 word analysis using the actual metric results from this notebook run.

In [ ]:
# Calculate differences from the baseline.
lstm_mae_change = lstm_result["MAE"] - baseline_result["MAE"]
attention_mae_change = attention_result["MAE"] - baseline_result["MAE"]
attention_vs_lstm = attention_result["MAE"] - lstm_result["MAE"]

# Use simple wording that remains honest for any result.
if best_network["MAE"] < baseline_result["MAE"]:
    baseline_statement = (
        f"The best learned model was {best_network['model']}. Its MAE was "
        f"{best_network['MAE']:.3f}, compared with {baseline_result['MAE']:.3f} for "
        f"the seasonal-naive baseline, so MAE improved by "
        f"{baseline_result['MAE'] - best_network['MAE']:.3f}."
    )
else:
    baseline_statement = (
        f"The seasonal-naive baseline remained strongest. Its MAE was "
        f"{baseline_result['MAE']:.3f}, while the best learned model, "
        f"{best_network['model']}, obtained {best_network['MAE']:.3f}."
    )

if attention_result["MAE"] < lstm_result["MAE"]:
    attention_statement = (
        f"Attention reduced MAE by {lstm_result['MAE'] - attention_result['MAE']:.3f} "
        "compared with the plain LSTM, so the extra mechanism produced a measurable benefit."
    )
else:
    attention_statement = (
        f"Attention changed MAE by {attention_vs_lstm:+.3f} compared with the plain LSTM, "
        "so the extra parameters did not provide a clear improvement on this smooth series."
    )

analysis = f"""Analysis

{baseline_statement} The corresponding best-model RMSE was {best_network['RMSE']:.3f} and MAPE was {best_network['MAPE']:.2f}%. {attention_statement} Attention may be more useful for longer forecast horizons, irregular seasonal patterns, or additional drivers such as weather and holidays. The split was chronological, so later observations were never used to train models that predicted earlier observations. Normalization statistics were calculated from the training inputs only, which prevented information from validation or test periods leaking backward. The test set was kept separate and scored after training. Both neural networks used direct multi-output forecasting: one 168-hour input produced all 24 predictions at once, so teacher forcing was not required. An autoregressive decoder would use its own earlier predictions during inference, which could create exposure bias and allow errors to accumulate. The forecast plot shows whether the selected network follows the daily rise and fall more closely than the seasonal rule. A useful next step would be to add hour-of-day, day-of-week, holiday, and temperature features, then test a longer horizon and prediction intervals."""

print(analysis)
print("
Word count:", len(analysis.split()))

# Save both trained models.
lstm_model.save("day22_lstm.keras")
attention_model.save("day22_lstm_attention.keras")
print("Saved day22_lstm.keras and day22_lstm_attention.keras")

## Final checklist
- Synthetic hourly energy-demand series generated offline
- 168-hour lookback and 24-hour horizon
- Chronological 70% / 15% / 15% split
- Train-only normalization
- Seasonal-naive baseline
- Plain LSTM model
- LSTM with self-attention
- MAE, RMSE, and MAPE comparison
- 24-hour prediction plot
- Teacher-forcing and autoregressive explanation
- 150–250 word result-based analysis

Run all cells, review the printed analysis, and save the executed notebook before submission.